## Saving as PyTorch Tensors
I'm going to re-do the DMS preembedding. Currently, it takes up a large amount of storage and takes a long time to load + overhead. Going to try a different method to try to get a more efficient approach to saving embedded sequences.

### ESM AA Embeddings

In [20]:
import os
import sys
import tqdm
import torch
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, EsmModel

from pnlp.embedding.tokenizer import ProteinTokenizer
from pnlp.embedding.nlp_embedding import NLPEmbedding
from pnlp.model.bert import BERT

In [21]:
# Parquet Making
class DMSDataset(Dataset):
    """ DMS virus sequence dataset. """

    def __init__(self, csv_file:str):
        self.df = pd.read_csv(csv_file, header=0, na_filter=False)
        self.max_sequence_length = self.df['sequence'].apply(len).max()

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx):
        columns = (
            self.df.iloc[idx]["label"],
            self.df.iloc[idx]["target"],
            self.df.iloc[idx]["sequence"],
            self.df.iloc[idx]["ACE2-binding_affinity"],
            self.df.iloc[idx]["RBD_expression"],
        )
        return columns

class ESM(nn.Module):
    def __init__(self, esm):
        super().__init__()
        self.esm = esm

    def forward(self, tokenized_seqs):
        with torch.set_grad_enabled(self.training):  # Enable gradients, managed by model.eval() or model.train() in epoch_iteration
            esm_last_hidden_state = self.esm(**tokenized_seqs).last_hidden_state # shape: (batch_size, sequence_length, embedding_dim)
            esm_aa_embedding = esm_last_hidden_state[:, 1:-1, :] # Amino Acid-level representations, [batch_size, sequence_length-2, embedding_dim], excludes 1st and last tokens
        return esm_aa_embedding

def run_model(model, tokenizer, dataloader, device, csv_file):   
    """ Call the ESM model to generate hidden states and store batch data in DataFrame directly. """
    
    model = model.to(device)
    model.eval()

    # Set the tqdm progress bar
    data_iter = tqdm.tqdm(enumerate(dataloader),
                          total = len(dataloader),
                          bar_format='{l_bar}{r_bar}')

    batch_dataframes = []
    all_embeddings = []

    with torch.no_grad():
        for _, batch_data in data_iter:
            labels, targets, sequences, binding_scores, expression_scores = batch_data 

            # Add 2 to max_length to account for additional tokens added to beginning and end by ESM
            max_length = dataloader.dataset.max_sequence_length + 2
            tokenized_seqs = tokenizer(sequences, return_tensors='pt', padding='max_length', max_length=max_length).to(device) 
            embeddings = model(tokenized_seqs) # shape: [batch_size, sequence_len, embedding_dim]

            # Create a DataFrame for the batch
            batch_df = pd.DataFrame({
                "label": labels,
                "target": targets,
                "ACE2-binding_affinity": binding_scores,
                "RBD_expression": expression_scores
            })
            batch_dataframes.append(batch_df)
            all_embeddings.append(embeddings.cpu())

        # Concatenate 
        result_df = pd.concat(batch_dataframes, ignore_index=True)
        all_embeddings = torch.cat(all_embeddings, dim=0)
        
        # Save data to a PyTorch tensor file
        save_as = csv_file.replace(".csv", "_ESM-AA-embedded.pt")
        save_as = save_as.replace("/data/dms", "/data/dms/pt")

        torch.save({
            "label": result_df["label"].tolist(),  # Use result_df here to get all labels
            "target": result_df["target"].tolist(),
            "ACE2-binding_affinity": result_df["ACE2-binding_affinity"].tolist(),
            "RBD_expression": result_df["RBD_expression"].tolist(),
            "embedding": all_embeddings
        }, save_as)
        print(f"Data saved to {save_as}")

In [22]:
# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
train_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_train.csv")
train_dataset = DMSDataset(train_csv_file)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# ESM input
esm_version = "facebook/esm2_t6_8M_UR50D"
esm = EsmModel.from_pretrained(esm_version, cache_dir='../../../model_downloads').to(device)
tokenizer = AutoTokenizer.from_pretrained(esm_version, cache_dir='../../../model_downloads')

model = ESM(esm)
run_model(model, tokenizer, train_dataloader, device, train_csv_file)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|| 1285/1285 [01:38<00:00, 13.01it/s]


Data saved to ../../data/dms/pt/mutation_combined_DMS_OLD_train_ESM-AA-embedded.pt


This is already about 12m faster to create the embeddings...

In [23]:
class DMSEmbeddedDataset(Dataset):
    """ Binding or Expression DMS Embedded Dataset, single target. """
    
    def __init__(self, tensor_file:str, result_tag:str):
        try:
            self.data = torch.load(tensor_file)
            self.target = 'ACE2-binding_affinity' if 'binding' in result_tag else 'RBD_expression'
        except (FileNotFoundError, KeyError, Exception) as e:
            print(f"Error reading the tensor file: {tensor_file}\n{e}", file=sys.stderr)
            sys.exit(1)

    def __len__(self) -> int:
        return len(self.data['label'])

    def __getitem__(self, idx):
        # label, seq, target
        return self.data['label'][idx], self.data['embedding'][idx], self.data[self.target][idx]
    
# Load in the parquets
data_dir = "../../data/dms/pt"
embedded_train = os.path.join(data_dir, "mutation_combined_DMS_OLD_train_ESM-AA-embedded.pt")
train_loader = DMSEmbeddedDataset(embedded_train, "binding")
print("Loaded training binding dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

train_loader = DMSEmbeddedDataset(embedded_train, "expression")
print("\nLoaded training expression dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_loader[i]
    print(f"{label}, {embedding.shape}, {target}")


Loaded training binding dataset from pt:
SARS-CoV-2-Y123W_P161T, torch.Size([201, 320]), 8.62
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([201, 320]), 9.78
SARS-CoV-2-E76S_N130F_G146W, torch.Size([201, 320]), 6.0
SARS-CoV-2-G51R_N107I, torch.Size([201, 320]), 8.6
SARS-CoV-2-Y91C, torch.Size([201, 320]), 10.364166666666668

Loaded training expression dataset from pt:
SARS-CoV-2-Y123W_P161T, torch.Size([201, 320]), 7.39
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([201, 320]), 8.12
SARS-CoV-2-E76S_N130F_G146W, torch.Size([201, 320]), 7.45
SARS-CoV-2-G51R_N107I, torch.Size([201, 320]), 8.07
SARS-CoV-2-Y91C, torch.Size([201, 320]), 9.300833333333332


This is a lot faster! The other took 3m 13.7s compared to the 12.4s here (for binding section). Let's re-do the rest of the embeddings.

In [24]:
# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
test_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_test.csv")
test_dataset = DMSDataset(test_csv_file)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# ESM input
esm_version = "facebook/esm2_t6_8M_UR50D"
esm = EsmModel.from_pretrained(esm_version, cache_dir='../../../model_downloads').to(device)
tokenizer = AutoTokenizer.from_pretrained(esm_version, cache_dir='../../../model_downloads')

model = ESM(esm)
run_model(model, tokenizer, test_dataloader, device, test_csv_file)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|| 322/322 [00:23<00:00, 13.84it/s]


Data saved to ../../data/dms/pt/mutation_combined_DMS_OLD_test_ESM-AA-embedded.pt


In [ ]:
# Load in the parquets
data_dir = "../../data/dms/pt"
embedded_test = os.path.join(data_dir, "mutation_combined_DMS_OLD_test_ESM-AA-embedded.pt")
test_loader = DMSEmbeddedDataset(embedded_test, "binding")
print("\nLoaded test binding dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = test_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

test_loader = DMSEmbeddedDataset(embedded_test, "expression")
print("\nLoaded test expression dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = test_loader[i]
    print(f"{label}, {embedding.shape}, {target}")


Loaded test binding dataset from pt:
SARS-CoV-2-T63S_Y121P_T201S, torch.Size([201, 320]), 9.38
SARS-CoV-2-S36E_Y143L, torch.Size([201, 320]), 10.27
SARS-CoV-2-Y39L_F99C, torch.Size([201, 320]), 8.985
SARS-CoV-2-V11C_I104Y, torch.Size([201, 320]), 9.59
SARS-CoV-2-K56L_D90T, torch.Size([201, 320]), 10.29

Loaded test expression dataset from pt:
SARS-CoV-2-T63S_Y121P_T201S, torch.Size([201, 320]), 9.07
SARS-CoV-2-S36E_Y143L, torch.Size([201, 320]), 9.85
SARS-CoV-2-Y39L_F99C, torch.Size([201, 320]), 8.280000000000001
SARS-CoV-2-V11C_I104Y, torch.Size([201, 320]), 7.68
SARS-CoV-2-K56L_D90T, torch.Size([201, 320]), 9.36


### ESM CLS Embeddings

In [17]:
# Parquet Making
class ESM(nn.Module):
    def __init__(self, esm):
        super().__init__()
        self.esm = esm

    def forward(self, tokenized_seqs):
        with torch.set_grad_enabled(self.training):  # Enable gradients, managed by model.eval() or model.train() in epoch_iteration
            esm_last_hidden_state = self.esm(**tokenized_seqs).last_hidden_state # shape: (batch_size, sequence_length, embedding_dim)
            esm_cls_embedding = esm_last_hidden_state[:, 0, :]  # CLS token embedding (sequence-level representations)
        return esm_cls_embedding

def run_model(model, tokenizer, dataloader, device, csv_file):   
    """ Call the ESM model to generate hidden states and store batch data in DataFrame directly. """
    
    model = model.to(device)
    model.eval()

    # Set the tqdm progress bar
    data_iter = tqdm.tqdm(enumerate(dataloader),
                          total = len(dataloader),
                          bar_format='{l_bar}{r_bar}')

    batch_dataframes = []
    all_embeddings = []

    with torch.no_grad():
        for _, batch_data in data_iter:
            labels, targets, sequences, binding_scores, expression_scores = batch_data 

            # Add 2 to max_length to account for additional tokens added to beginning and end by ESM
            max_length = dataloader.dataset.max_sequence_length + 2
            tokenized_seqs = tokenizer(sequences, return_tensors='pt', padding='max_length', max_length=max_length).to(device) 
            embeddings = model(tokenized_seqs) # shape: [batch_size, sequence_len, embedding_dim]

            # Create a DataFrame for the batch
            batch_df = pd.DataFrame({
                "label": labels,
                "target": targets,
                "ACE2-binding_affinity": binding_scores,
                "RBD_expression": expression_scores
            })
            batch_dataframes.append(batch_df)
            all_embeddings.append(embeddings.cpu())

        # Concatenate 
        result_df = pd.concat(batch_dataframes, ignore_index=True)
        all_embeddings = torch.cat(all_embeddings, dim=0)
        
        # Save data to a PyTorch tensor file
        save_as = csv_file.replace(".csv", "_ESM-CLS-embedded.pt")
        save_as = save_as.replace("/data/dms", "/data/dms/pt")

        torch.save({
            "label": result_df["label"].tolist(),  # Use result_df here to get all labels
            "target": result_df["target"].tolist(),
            "ACE2-binding_affinity": result_df["ACE2-binding_affinity"].tolist(),
            "RBD_expression": result_df["RBD_expression"].tolist(),
            "embedding": all_embeddings
        }, save_as)
        print(f"Data saved to {save_as}")

In [27]:
# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
train_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_train.csv")
train_dataset = DMSDataset(train_csv_file)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# ESM input
esm_version = "facebook/esm2_t6_8M_UR50D"
esm = EsmModel.from_pretrained(esm_version, cache_dir='../../../model_downloads').to(device)
tokenizer = AutoTokenizer.from_pretrained(esm_version, cache_dir='../../../model_downloads')

model = ESM(esm)
run_model(model, tokenizer, train_dataloader, device, train_csv_file)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|| 1285/1285 [01:28<00:00, 14.51it/s]


Data saved to ../../data/dms/pt/mutation_combined_DMS_OLD_train_ESM-CLS-embedded.pt


In [28]:
# Load in the parquets
data_dir = "../../data/dms/pt"
embedded_train = os.path.join(data_dir, "mutation_combined_DMS_OLD_train_ESM-CLS-embedded.pt")
train_loader = DMSEmbeddedDataset(embedded_train, "binding")
print("Loaded training binding dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

train_loader = DMSEmbeddedDataset(embedded_train, "expression")
print("\nLoaded training expression dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

Loaded training binding dataset from pt:
SARS-CoV-2-Y123W_P161T, torch.Size([320]), 8.62
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([320]), 9.78
SARS-CoV-2-E76S_N130F_G146W, torch.Size([320]), 6.0
SARS-CoV-2-G51R_N107I, torch.Size([320]), 8.6
SARS-CoV-2-Y91C, torch.Size([320]), 10.364166666666668

Loaded training expression dataset from pt:
SARS-CoV-2-Y123W_P161T, torch.Size([320]), 7.39
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([320]), 8.12
SARS-CoV-2-E76S_N130F_G146W, torch.Size([320]), 7.45
SARS-CoV-2-G51R_N107I, torch.Size([320]), 8.07
SARS-CoV-2-Y91C, torch.Size([320]), 9.300833333333332


In [29]:
# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
test_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_test.csv")
test_dataset = DMSDataset(test_csv_file)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# ESM input
esm_version = "facebook/esm2_t6_8M_UR50D"
esm = EsmModel.from_pretrained(esm_version, cache_dir='../../../model_downloads').to(device)
tokenizer = AutoTokenizer.from_pretrained(esm_version, cache_dir='../../../model_downloads')

model = ESM(esm)
run_model(model, tokenizer, test_dataloader, device, test_csv_file)

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|| 322/322 [00:22<00:00, 14.49it/s]

Data saved to ../../data/dms/pt/mutation_combined_DMS_OLD_test_ESM-CLS-embedded.pt


In [30]:
# Load in the parquets
data_dir = "../../data/dms/pt"
embedded_test = os.path.join(data_dir, "mutation_combined_DMS_OLD_test_ESM-CLS-embedded.pt")
test_loader = DMSEmbeddedDataset(embedded_test, "binding")
print("\nLoaded test binding dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = test_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

test_loader = DMSEmbeddedDataset(embedded_test, "expression")
print("\nLoaded test expression dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = test_loader[i]
    print(f"{label}, {embedding.shape}, {target}")


Loaded test binding dataset from pt:
SARS-CoV-2-T63S_Y121P_T201S, torch.Size([320]), 9.38
SARS-CoV-2-S36E_Y143L, torch.Size([320]), 10.27
SARS-CoV-2-Y39L_F99C, torch.Size([320]), 8.985
SARS-CoV-2-V11C_I104Y, torch.Size([320]), 9.59
SARS-CoV-2-K56L_D90T, torch.Size([320]), 10.29

Loaded test expression dataset from pt:
SARS-CoV-2-T63S_Y121P_T201S, torch.Size([320]), 9.07
SARS-CoV-2-S36E_Y143L, torch.Size([320]), 9.85
SARS-CoV-2-Y39L_F99C, torch.Size([320]), 8.280000000000001
SARS-CoV-2-V11C_I104Y, torch.Size([320]), 7.68
SARS-CoV-2-K56L_D90T, torch.Size([320]), 9.36


### NLP Embeddings

In [31]:
def run_model(model, tokenizer, dataloader, device, csv_file):   
    """ Call the model to generate hidden states and store batch data in DataFrame directly. """
    
    model = model.to(device)
    model.eval()

    # Set the tqdm progress bar
    data_iter = tqdm.tqdm(enumerate(dataloader),
                          total = len(dataloader),
                          bar_format='{l_bar}{r_bar}')

    batch_dataframes = []
    all_embeddings = []

    with torch.no_grad():
        for _, batch_data in data_iter:
            labels, targets, sequences, binding_scores, expression_scores = batch_data 
            tokenized_seqs = tokenizer(sequences).to(device) 
            embeddings, _ = model(tokenized_seqs) # shape: [batch_size, seq_length embedding_dim]

            # Create a DataFrame for the batch
            batch_df = pd.DataFrame({
                "label": labels,
                "target": targets,
                "ACE2-binding_affinity": binding_scores,
                "RBD_expression": expression_scores
            })
            batch_dataframes.append(batch_df)
            all_embeddings.append(embeddings.cpu())

        # Concatenate 
        result_df = pd.concat(batch_dataframes, ignore_index=True)
        all_embeddings = torch.cat(all_embeddings, dim=0)
        
        # Save data to a PyTorch tensor file
        save_as = csv_file.replace(".csv", "_NLP-embedded.pt")
        save_as = save_as.replace("/data/dms", "/data/dms/pt")

        torch.save({
            "label": result_df["label"].tolist(),  # Use result_df here to get all labels
            "target": result_df["target"].tolist(),
            "ACE2-binding_affinity": result_df["ACE2-binding_affinity"].tolist(),
            "RBD_expression": result_df["RBD_expression"].tolist(),
            "embedding": all_embeddings
        }, save_as)
        print(f"Data saved to {save_as}")
        

In [32]:
# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
train_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_train.csv")
train_dataset = DMSDataset(train_csv_file)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# NLP input
max_len = 280
embedding_dim = 320
dropout = 0.1
mask_prob=0

model = NLPEmbedding(embedding_dim, max_len, dropout)
tokenizer = ProteinTokenizer(max_len, mask_prob)

# Load NLP weights from BERT model
bert_model_pth = "../../results/run_results/bert_mlm-esm_init/bert_mlm-esm_init-RBD-2024-09-25_20-29/best_saved_model.pth"
saved_state = torch.load(bert_model_pth, map_location=device, weights_only=False)
model_state = saved_state['model_state_dict']
embedding_weights = model_state['bert.embedding.token_embedding.weight']
with torch.no_grad():
    model.token_embedding.weight = nn.Parameter(embedding_weights)

run_model(model, tokenizer, train_dataloader, device, train_csv_file)

100%|| 1285/1285 [00:43<00:00, 29.79it/s]


Data saved to ../../data/dms/pt/mutation_combined_DMS_OLD_train_NLP-embedded.pt


In [33]:
# Load in the parquets
data_dir = "../../data/dms/pt"
embedded_train = os.path.join(data_dir, "mutation_combined_DMS_OLD_train_NLP-embedded.pt")
train_loader = DMSEmbeddedDataset(embedded_train, "binding")
print("Loaded training binding dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

train_loader = DMSEmbeddedDataset(embedded_train, "expression")
print("\nLoaded training expression dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = train_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

Loaded training binding dataset from pt:
SARS-CoV-2-Y123W_P161T, torch.Size([201, 320]), 8.62
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([201, 320]), 9.78
SARS-CoV-2-E76S_N130F_G146W, torch.Size([201, 320]), 6.0
SARS-CoV-2-G51R_N107I, torch.Size([201, 320]), 8.6
SARS-CoV-2-Y91C, torch.Size([201, 320]), 10.364166666666668

Loaded training expression dataset from pt:
SARS-CoV-2-Y123W_P161T, torch.Size([201, 320]), 7.39
SARS-CoV-2-Q84F_N110D_E141N_P149F, torch.Size([201, 320]), 8.12
SARS-CoV-2-E76S_N130F_G146W, torch.Size([201, 320]), 7.45
SARS-CoV-2-G51R_N107I, torch.Size([201, 320]), 8.07
SARS-CoV-2-Y91C, torch.Size([201, 320]), 9.300833333333332


In [34]:
# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
batch_size = 64

# Data file
data_dir = "../../data/dms"
test_csv_file = os.path.join(data_dir, "mutation_combined_DMS_OLD_test.csv")
test_dataset = DMSDataset(test_csv_file)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

# NLP input
max_len = 280
embedding_dim = 320
dropout = 0.1
mask_prob=0

model = NLPEmbedding(embedding_dim, max_len, dropout)
tokenizer = ProteinTokenizer(max_len, mask_prob)

# Load NLP weights from BERT model
bert_model_pth = "../../results/run_results/bert_mlm-esm_init/bert_mlm-esm_init-RBD-2024-09-25_20-29/best_saved_model.pth"
saved_state = torch.load(bert_model_pth, map_location=device, weights_only=False)
model_state = saved_state['model_state_dict']
embedding_weights = model_state['bert.embedding.token_embedding.weight']
with torch.no_grad():
    model.token_embedding.weight = nn.Parameter(embedding_weights)

run_model(model, tokenizer, test_dataloader, device, test_csv_file)

100%|| 322/322 [00:14<00:00, 22.07it/s]


Data saved to ../../data/dms/pt/mutation_combined_DMS_OLD_test_NLP-embedded.pt


In [35]:
# Load in the parquets
data_dir = "../../data/dms/pt"
embedded_test = os.path.join(data_dir, "mutation_combined_DMS_OLD_test_NLP-embedded.pt")
test_loader = DMSEmbeddedDataset(embedded_test, "binding")
print("\nLoaded test binding dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = test_loader[i]
    print(f"{label}, {embedding.shape}, {target}")

test_loader = DMSEmbeddedDataset(embedded_test, "expression")
print("\nLoaded test expression dataset from pt:")
for i in range(5):  # adjust the range as needed to print a few examples
    label, embedding, target = test_loader[i]
    print(f"{label}, {embedding.shape}, {target}")


Loaded test binding dataset from pt:
SARS-CoV-2-T63S_Y121P_T201S, torch.Size([201, 320]), 9.38
SARS-CoV-2-S36E_Y143L, torch.Size([201, 320]), 10.27
SARS-CoV-2-Y39L_F99C, torch.Size([201, 320]), 8.985
SARS-CoV-2-V11C_I104Y, torch.Size([201, 320]), 9.59
SARS-CoV-2-K56L_D90T, torch.Size([201, 320]), 10.29

Loaded test expression dataset from pt:
SARS-CoV-2-T63S_Y121P_T201S, torch.Size([201, 320]), 9.07
SARS-CoV-2-S36E_Y143L, torch.Size([201, 320]), 9.85
SARS-CoV-2-Y39L_F99C, torch.Size([201, 320]), 8.280000000000001
SARS-CoV-2-V11C_I104Y, torch.Size([201, 320]), 7.68
SARS-CoV-2-K56L_D90T, torch.Size([201, 320]), 9.36
